# Other utilities

## Typed callbacks

Q-learning and SARSA accept one callback or an ordered callback iterable. EpisodeContext and TransitionContext expose read-only lifecycle data without requiring an RL subclass.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt

from bettermdptools.algorithms.rl import RL
from bettermdptools.utils.callbacks import Callbacks, EpisodeContext, TransitionContext
from bettermdptools.utils.plots import Plots
from bettermdptools.utils.test_env import TestEnv


class Metrics(Callbacks):
    def __init__(self):
        self.transitions = []
        self.episode_returns = []

    def on_env_step(self, caller, *, context: TransitionContext):
        self.transitions.append((context.episode, context.action, context.reward))

    def on_episode_end(self, caller, *, context: EpisodeContext):
        self.episode_returns.append(context.total_reward)


env = gym.make("FrozenLake-v1", is_slippery=False, max_episode_steps=4)
metrics = Metrics()
Q, V, pi, Q_track, pi_track, rewards = RL(
    env, callbacks=metrics
).q_learning(n_episodes=3, seed=7)
metrics.episode_returns

## Callable policies

TestEnv keeps indexable policies and also accepts callables with a state-only or state-plus-info signature.

In [ ]:
def learned_policy(state, info):
    return pi[state]


scores = TestEnv.test_env(
    env, n_iters=2, pi=learned_policy, seed=7
)
scores

## Composable plotting

Pure transformations prepare plot data. Rendering targets caller-owned axes and returns each axes handle without changing the global plotting theme.

In [ ]:
actions = {
    0: "MOVE LEFT",
    1: "MOVE DOWN",
    2: "MOVE RIGHT",
    3: "MOVE UP",
}
mapped_values, policy_labels = Plots.get_policy_map(
    pi, V, actions, (4, 4)
)

figure, (value_ax, policy_ax) = plt.subplots(1, 2, figsize=(12, 5))
Plots.values_heat_map(
    V, "State values", (4, 4), show=False, ax=value_ax
)
Plots.plot_policy(
    mapped_values,
    policy_labels,
    (4, 4),
    "Learned policy",
    show=False,
    ax=policy_ax,
)
figure.tight_layout()
env.close()
figure